In [4]:
# ============================================================
# RETINAGENT — CONSISTENCY CHECKER AGENT
# ============================================================

import os
import torch


# ------------------------------------------------------------
# 1. Locate GraderAgent blackboard from Utility Script
# ------------------------------------------------------------

def find_grader_blackboard():
    candidates = []

    for root, dirs, files in os.walk("/kaggle/usr/lib"):
        for file in files:
            if file == "grader_blackboard.pt":
                candidates.append(
                    os.path.join(root, file)
                )

    if not candidates:
        raise FileNotFoundError(
            "grader_blackboard.pt not found in /kaggle/usr/lib.\n"
            "Make sure the Grader_Agent notebook is added to the "
            "RetinaAgent Collection and its output is available."
        )

    return candidates[0]


GRADER_BB_PATH = find_grader_blackboard()

print("=" * 70)
print("RETINAGENT - CONSISTENCY CHECKER AGENT")
print("=" * 70)

print("\nGrader blackboard found:")
print(GRADER_BB_PATH)


RETINAGENT - CONSISTENCY CHECKER AGENT

Grader blackboard found:
/kaggle/usr/lib/notebooks/divyanshukj4495/grader_agent/grader_blackboard.pt


In [5]:
# ------------------------------------------------------------
# 2. Load Grader Blackboard
# ------------------------------------------------------------

grader_blackboard = torch.load(
    GRADER_BB_PATH,
    map_location="cpu",
    weights_only=False
)

if "output" not in grader_blackboard:
    raise KeyError(
        "Grader blackboard does not contain 'output'."
    )

grader_output = grader_blackboard["output"]

print("\nGrader output loaded.")


# ------------------------------------------------------------
# 3. Validate Grader Output
# ------------------------------------------------------------

required_fields = {
    "grade",
    "grade_label",
    "confidence",
    "tta_grades",
    "tta_confidences",
    "tta_spread",
    "clinical_missing_fields",
    "fusion_status",
    "fusion_method",
}

missing_fields = required_fields - set(grader_output.keys())

if missing_fields:
    raise KeyError(
        f"Missing Grader fields: {sorted(missing_fields)}"
    )

print("Grader fields validated.")



Grader output loaded.
Grader fields validated.


In [6]:
# ------------------------------------------------------------
# 4. Extract Grader Information
# ------------------------------------------------------------

final_grade = int(grader_output["grade"])

grade_label = grader_output["grade_label"]

confidence = float(
    grader_output["confidence"]
)

tta_grades = [
    int(x)
    for x in grader_output["tta_grades"]
]

tta_confidences = [
    float(x)
    for x in grader_output["tta_confidences"]
]

clinical_missing_fields = list(
    grader_output["clinical_missing_fields"]
)

fusion_status = grader_output["fusion_status"]

fusion_method = grader_output["fusion_method"]


In [7]:
# ------------------------------------------------------------
# 5. Validate TTA
# ------------------------------------------------------------

if len(tta_grades) != 3:
    raise ValueError(
        f"Expected 3 TTA predictions, "
        f"got {len(tta_grades)}."
    )

if len(tta_confidences) != 3:
    raise ValueError(
        f"Expected 3 TTA confidences, "
        f"got {len(tta_confidences)}."
    )


# ------------------------------------------------------------
# 6. Calculate TTA Spread
# ------------------------------------------------------------

tta_spread = (
    max(tta_grades)
    -
    min(tta_grades)
)

stored_spread = int(
    grader_output["tta_spread"]
)

if tta_spread != stored_spread:
    raise ValueError(
        "TTA spread mismatch between stored "
        "Grader value and recalculated value."
    )


In [8]:
# ------------------------------------------------------------
# 7. TTA Consistency Rule
# ------------------------------------------------------------
#
# Paper:
#
# max(TTA grade) - min(TTA grade) > 1
#              ↓
#       clinician review
#
# ------------------------------------------------------------

if tta_spread > 1:

    tta_consistent = False

    review_required = True

    consistency_status = "INCONSISTENT"

    review_reason = (
        "TTA grade spread exceeds 1."
    )

else:

    tta_consistent = True

    review_required = False

    consistency_status = "CONSISTENT"

    review_reason = None


# ------------------------------------------------------------
# 8. Prediction Agreement
# ------------------------------------------------------------

unique_grades = sorted(
    set(tta_grades)
)

if len(unique_grades) == 1:

    prediction_agreement = "EXACT"

elif tta_spread <= 1:

    prediction_agreement = "WITHIN_ONE_GRADE"

else:

    prediction_agreement = "DISAGREEMENT"


In [9]:
# ------------------------------------------------------------
# 9. Confidence Status
# ------------------------------------------------------------

if confidence >= 0.70:

    confidence_status = "HIGH"

elif confidence >= 0.50:

    confidence_status = "MODERATE"

else:

    confidence_status = "LOW"


# ------------------------------------------------------------
# 10. Clinical Completeness
# ------------------------------------------------------------

missing_count = len(
    clinical_missing_fields
)

if missing_count == 0:

    clinical_completeness = "COMPLETE"

elif missing_count < 7:

    clinical_completeness = "PARTIALLY_COMPLETE"

else:

    clinical_completeness = "LARGELY_MISSING"


In [10]:
# ------------------------------------------------------------
# 11. Build Consistency Report
# ------------------------------------------------------------

consistency_report = {

    "agent": "ConsistencyCheckerAgent",

    # Final grading
    "grade": final_grade,
    "grade_label": grade_label,
    "confidence": confidence,

    # TTA consistency
    "tta_grades": tta_grades,
    "tta_confidences": tta_confidences,
    "tta_spread": tta_spread,

    "tta_consistent": tta_consistent,
    "consistency_status": consistency_status,

    "prediction_agreement": prediction_agreement,

    # Confidence
    "confidence_status": confidence_status,

    # Clinical information
    "clinical_missing_fields": clinical_missing_fields,
    "clinical_missing_count": missing_count,
    "clinical_completeness": clinical_completeness,

    # Fusion information
    "fusion_status": fusion_status,
    "fusion_method": fusion_method,

    # Review
    "review_required": review_required,
    "review_reason": review_reason,
}


# ------------------------------------------------------------
# 12. Save Consistency Blackboard
# ------------------------------------------------------------

CONSISTENCY_BB_PATH = (
    "/kaggle/working/consistency_blackboard.pt"
)

consistency_blackboard = {
    "agent": "ConsistencyCheckerAgent",
    "output": consistency_report,
}

torch.save(
    consistency_blackboard,
    CONSISTENCY_BB_PATH
)


In [11]:
# ------------------------------------------------------------
# 13. Display Report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSISTENCY CHECK REPORT")
print("=" * 70)

print(
    f"\nFinal grade: "
    f"{final_grade} ({grade_label})"
)

print(
    f"Confidence: "
    f"{confidence:.4f} ({confidence_status})"
)

print(
    "\nTTA grades:",
    tta_grades
)

print(
    "TTA confidences:",
    [round(x, 4) for x in tta_confidences]
)

print(
    "TTA spread:",
    tta_spread
)

print(
    "Consistency status:",
    consistency_status
)

print(
    "Prediction agreement:",
    prediction_agreement
)

print(
    "\nClinical completeness:",
    clinical_completeness
)

print(
    "Missing clinical fields:",
    missing_count
)

if clinical_missing_fields:
    print(
        "Missing:",
        clinical_missing_fields
    )

print(
    "\nFusion status:",
    fusion_status
)

print(
    "Fusion method:",
    fusion_method
)

print(
    "\nReview required:",
    review_required
)

if review_reason:
    print(
        "Review reason:",
        review_reason
    )

print("\n" + "=" * 70)

print(
    "Consistency blackboard saved locally:"
)

print(CONSISTENCY_BB_PATH)

print("=" * 70)

print("\nRETINAGENT CONSISTENCY CHECKER COMPLETE.")



CONSISTENCY CHECK REPORT

Final grade: 0 (No DR)
Confidence: 0.3649 (LOW)

TTA grades: [0, 0, 0]
TTA confidences: [0.3649, 0.3355, 0.415]
TTA spread: 0
Consistency status: CONSISTENT
Prediction agreement: EXACT

Clinical completeness: LARGELY_MISSING
Missing clinical fields: 7
Missing: ['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms']

Fusion status: IMAGE_GROUNDED_NO_TRAINED_FUSION
Fusion method: image_grade_with_clinical_context

Review required: False

Consistency blackboard saved locally:
/kaggle/working/consistency_blackboard.pt

RETINAGENT CONSISTENCY CHECKER COMPLETE.
